In [1]:
import os, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from transformers import ASTFeatureExtractor, ASTModel
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- CONFIG ---
AST_WEIGHTS_PATH = '/kaggle/input/datasets/neerajsurin/best-models/best_ast.pth'
RESNET_WEIGHTS_PATH = '/kaggle/input/datasets/neerajsurin/best-models/resnet50d_best.pth'

TEST_CSV = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/test.csv'
MASHUPS_DIR = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/mashups'
OUT_DIR = '/kaggle/working'

SR = 16000
N_SAMPLES = SR * 10
FULL_SAMPLES = SR * 30
TTA_CROPS = 11

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
I2G = {i: g for i, g in enumerate(GENRES)}

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF Auth OK")

# ==========================================
# 1. AST MODEL DEFINITION (From your 0.981 code)
# ==========================================
class Patchout(nn.Module):
    def __init__(self, p_time=0.4, p_freq=0.2):
        super().__init__()
        self.p_t, self.p_f = p_time, p_freq
    def forward(self, x):
        return x

class ASTClassifier(nn.Module):
    def __init__(self, n_cls=10):
        super().__init__()
        self.patchout = Patchout(0, 0) # Disabled for inference
        self.ast = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
        self.hidden = self.ast.config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(self.hidden),
            nn.Dropout(0.15),
            nn.Linear(self.hidden, 384),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(384, n_cls)
        )
    def forward(self, x):
        x = self.patchout(x)
        out = self.ast(x).pooler_output
        return self.head(out)

ast_processor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", sampling_rate=16000)

# ==========================================
# 2. RESNET MODEL DEFINITION (From your 0.968 code)
# ==========================================
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000, n_fft=1024, hop_length=320, f_min=20, f_max=8000, n_mels=128
        )
        self.a2db = torchaudio.transforms.AmplitudeToDB()

    def forward(self, x):
        with torch.cuda.amp.autocast(enabled=False): 
            x = x.float()
            x = self.mel(x)
            x = self.a2db(x)
            m = x.mean(dim=[-2, -1], keepdim=True)
            s = x.std(dim=[-2, -1], keepdim=True)
            x = (x - m) / (s + 1e-5)
            return x.unsqueeze(1)

resnet_fe = ResNetFeatureExtractor().to(device).eval()

# ==========================================
# 3. LOAD MODELS
# ==========================================
print("Loading Models...")
ast_model = ASTClassifier().to(device)
ast_ckpt = torch.load(AST_WEIGHTS_PATH, map_location=device)
ast_model.load_state_dict(ast_ckpt['state'] if 'state' in ast_ckpt else ast_ckpt)
ast_model.eval()

resnet_model = timm.create_model('resnet50d', pretrained=False, in_chans=1, num_classes=10).to(device)
resnet_ckpt = torch.load(RESNET_WEIGHTS_PATH, map_location=device)
resnet_model.load_state_dict(resnet_ckpt['state'] if 'state' in resnet_ckpt else resnet_ckpt)
resnet_model.eval()

# ==========================================
# 4. INFERENCE
# ==========================================
print("\nRunning Ensemble Inference...")
test_df = pd.read_csv(TEST_CSV)
all_preds = np.zeros((len(test_df), 10))

def get_audio(path):
    try:
        y, _ = librosa.load(path, sr=SR, duration=30, mono=True)
    except:
        y = np.zeros(FULL_SAMPLES, dtype=np.float32)
        
    if len(y) < FULL_SAMPLES:
        y = np.pad(y, (0, FULL_SAMPLES - len(y)))
    else:
        y = y[:FULL_SAMPLES]
        
    rms = np.sqrt(np.mean(y**2)) + 1e-8
    return np.clip(y * (10**(-20/20) / rms), -1.0, 1.0)

with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        audio = get_audio(f"{MASHUPS_DIR}/song{str(row['id']).zfill(4)}.wav")
        
        # Create TTA Crops
        max_st = FULL_SAMPLES - N_SAMPLES
        starts = np.linspace(0, max_st, TTA_CROPS, dtype=int)
        crops = [audio[s:s+N_SAMPLES] for s in starts]
        
        # --- AST Prediction ---
        ast_inputs = ast_processor(crops, sampling_rate=SR, return_tensors="pt").input_values.to(device)
        with torch.cuda.amp.autocast():
            ast_logits = ast_model(ast_inputs)
        ast_probs = F.softmax(ast_logits, dim=1).mean(dim=0).cpu().numpy()
        
        # --- ResNet Prediction ---
        resnet_inputs = torch.tensor(np.array(crops), dtype=torch.float32).to(device)
        resnet_feats = resnet_fe(resnet_inputs)
        with torch.cuda.amp.autocast():
            resnet_logits = resnet_model(resnet_feats)
        resnet_probs = F.softmax(resnet_logits, dim=1).mean(dim=0).cpu().numpy()
        
        # --- Ensemble Average ---
        all_preds[idx] = (0.6 * ast_probs) + (0.4 * resnet_probs)

# ==========================================
# 5. SAVE & ANALYZE
# ==========================================
test_df['genre'] = [I2G[i] for i in all_preds.argmax(axis=1)]
test_df['id'] = test_df['id'].astype(str).str.zfill(4)
test_df[['id', 'genre']].to_csv(f"{OUT_DIR}/submission.csv", index=False)

print("\n" + "="*55 + "\nPREDICTION ANALYSIS\n" + "="*55)
counts = test_df['genre'].value_counts()
print(f"\nDistribution:\n{counts.sort_index()}")

prob_df = pd.DataFrame(all_preds, columns=GENRES)
prob_df.insert(0, 'id', test_df['id'].values)
prob_df = prob_df.sort_values('id')
prob_df.to_csv(f"{OUT_DIR}/probs.csv", index=False)

max_probs = all_preds.max(axis=1)
print(f"\nConfidence: mean={max_probs.mean():.4f}, min={max_probs.min():.4f}")
print(f"  >0.95: {(max_probs > 0.95).sum()}")
print(f"  >0.90: {(max_probs > 0.90).sum()}")
print(f"  <0.50: {(max_probs < 0.50).sum()}")

print(f"\nBalance: {counts.min()} - {counts.max()} (std={counts.std():.1f})")

pred_genres = test_df['genre'].values
print("\nPer-genre confidence:")
print(f"{'Genre':<12} {'Count':>6} {'MeanConf':>9} {'MinConf':>9} {'<0.7':>6} {'<0.5':>6}")
print("-" * 55)
for g in GENRES:
    mask = pred_genres == g
    if mask.sum() > 0:
        g_confs = max_probs[mask]
        print(f"{g:<12} {mask.sum():>6} {g_confs.mean():>9.4f} {g_confs.min():>9.4f} "
              f"{(g_confs < 0.7).sum():>6} {(g_confs < 0.5).sum():>6}")

print("\nInference Complete!")

HF Auth OK


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Loading Models...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.dense.bias       | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Ensemble Inference...


  0%|          | 0/3020 [00:00<?, ?it/s]


PREDICTION ANALYSIS

Distribution:
genre
blues        324
classical    262
country      272
disco        309
hiphop       312
jazz         295
metal        317
pop          308
reggae       310
rock         311
Name: count, dtype: int64

Confidence: mean=0.7389, min=0.1772
  >0.95: 137
  >0.90: 650
  <0.50: 248

Balance: 262 - 324 (std=20.0)

Per-genre confidence:
Genre         Count  MeanConf   MinConf   <0.7   <0.5
-------------------------------------------------------
blues           324    0.7876    0.1772     80     37
classical       262    0.7188    0.2112    106     53
country         272    0.6684    0.2051    152     40
disco           309    0.7131    0.2830    152      7
hiphop          312    0.8995    0.3364     14      8
jazz            295    0.7999    0.2871     65     19
metal           317    0.7017    0.3533    163     16
pop             308    0.7306    0.2769    124     20
reggae          310    0.7312    0.2003    127     15
rock            311    0.6274    0.2